# B8 (الـlaminated seismic bearing): تقارب الشبكة على GPU -- داخل نطاق 10^5-10^6 عنصر

دراسة CPU السابقة أعطت تقارب حقيقي ونظيف (29.2%->21.2%->14.1%->7.0% لخطأ حقل Cauchy المحلي عند 576-3600 عنصر)، بس المرجع (5,184 عنصر) **مش مؤكد إنه متقارب** -- هاي الخلية بتكمل صح: بتحل مرجعين حقيقيين (القديم ~1.05 مليون عنصر، الجديد الأدق ~2.9 مليون عنصر) وبتقارنهم مباشرة **قبل** ما تعتبر أي وحدة منهم مرجع نهائي موثوق -- بالضبط نفس الفحص يلي انعمل لـB3.

**تعليمات صريحة انلتزم فيها هون**:
1. ما منسمي أي شبكة "مرجع متقارب" لحد ما فحص المرجع-مقابل-المرجع يثبت هيك فعليًا.
2. ما منستخدم ارتفاع أعلى إجهاد حقيقي (true_max) كدليل إنو B8 أصعب من B3 -- رقم بيرتفع مع الدقة ممكن يعني "لسا الشبكة ناقصة" بنفس قد ما يعني "فيه ميزة حادة فعلاً"، وما فيه طريقة تفرق بينهم من رقم واحد بس. true_max بيبقى تشخيصي بس، متل ما هو الحال دائمًا بـB1/B2/B3.
3. منطقة قياس الإجهاد (r=R_out, theta=0, منتصف أول شيم داخلي، نصف قطر المنطقة=6×سماكة الشيم) **ثابتة من هلق وطالعة**، ما بتتغير مع الدقة.
4. خطأ حقل Cauchy الإقليمي (region-Cauchy field error) هو الـQoI المحلي **الأساسي**، مش avg ولا true_max.

**السلّم**: نفس دقات الـCPU (576 لـ5,184 عنصر) + دقات GPU جديدة توصل لنطاق 10^5-10^6 (19k, 51k, 136k, 373k, 792k عنصر)، مقارنة بالمرجع الأدق (~2.9 مليون).

**الهدف المحدد**: هل خطأ الـQoI الإقليمي بيضل تقريبًا 5-10% داخل نطاق 10^5-10^6 عنصر؟ (متطلب تيمون بالضبط).

**الكلفة المتوقعة**: 12 حل (10 بالسلّم + مرجعين)، أكبرهم ~2.9 مليون عنصر -- أكبر بكتير من أكبر شبكة حُلّت لـB3 (424 ألف)، فمتوقع ياخد وقت أطول بكتير (ممكن ساعات على GPU عادي) -- إذا صار في مشكلة ذاكرة أو وقت، أول شي نجرب تصغير NEW_FINE_RESOLUTION بالخلية.

بالنسبة لقاعدة عمر الدائمة (2026-09-21): هاي الخلية بتولّد صورتين (فحص المرجعين، وملخص التقارب الكامل)، بتحفظهم على Drive **وكمان** بتعرضهم مباشرة بآخر الخلية.


In [ ]:
# =====================================================================
#  FROZEN ARCHIVE -- B8-prototype/pilot (2026-09-22/23). This is the
#  exact notebook that produced the real, GPU-confirmed pilot result
#  (region-Cauchy field error 7.53% at 791,864 elements, inside the
#  advisor's 10^5-10^6-element/5-10% target). Imports mesh_convergence_
#  B8_prototype.py (not the revised mesh_convergence_B8.py) so this
#  result stays independently reproducible while B8-final is built with
#  real published-source geometry/materials and a corrected stress QoI
#  region (see data_generate_B8.py's own docstring for what changed and
#  why). Do not edit this file.
#
#  CELL -- B8 (3D laminated annular elastomeric seismic bearing, Option
#  B) mesh convergence, extended to GPU resolutions into and above the
#  10^5-10^6-element range the advisor asked about.
#
#  Per the explicit instructions for this run (given after the CPU-only
#  study): (1) do NOT describe any mesh here as a "converged reference"
#  until a reference-to-reference comparison actually supports that;
#  (2) do NOT use a rising true (raw) peak stress as evidence that this
#  design is "harder" than B3 -- an increasing raw max with resolution
#  is exactly as consistent with "still refining, not yet converged" as
#  with "a genuinely sharp feature," and cannot be told apart from a
#  single rising number alone (this is exactly why B1/B2/B3 already
#  treat true_max as diagnostic-only, never a threshold QoI -- the same
#  discipline applies here); (3) the region-Cauchy-stress FIELD ERROR
#  (volume-weighted, quadrature-based, symmetric element-centroid
#  methodology already validated for B3) is the PRIMARY local QoI,
#  true_max is printed only as a secondary diagnostic; (4) the physical
#  stress-evaluation region (r=R_out, theta=0, z=first internal shim's
#  mid-height, region_radius=6*T_SHIM) is now FIXED and must not change
#  again as resolution increases; (5) extend the ladder into
#  approximately 10^5-10^6 elements, use an even finer mesh above that
#  range as the reference, and INCLUDE a reference-to-reference
#  comparison (old vs new fine reference) to show the chosen reference
#  is actually converged -- exactly the same check already done for
#  B3's own GPU study (cell_b3_gpu_mesh_convergence.py), reused here
#  verbatim in structure.
#
#  Per the 2026-09-21 standing rule: generates figures during the
#  analysis AND a final summary figure, saves all of them to Drive, and
#  displays them inline in this notebook's own output.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import json
import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if _mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.'):
        del sys.modules[_mod_name]

import numpy as np
import matplotlib.pyplot as plt
import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

# Real fix, not defensive boilerplate: confirmed directly that re-running
# this cell in the SAME Colab kernel after an earlier OOM does NOT free
# that earlier crash's GPU memory, even with gc.collect()/empty_cache()
# elsewhere in this cell -- because Jupyter/IPython automatically stores
# the last exception's full traceback (accessible as sys.last_traceback),
# and every local variable in every frame of that traceback (including
# the large GPU tensors alive at the moment of the crash) stays reachable
# -- and therefore un-collectable -- until that stored traceback itself
# is cleared. A true Runtime > Restart session clears it as a side effect
# of killing the process; simply re-running this cell does not. Clearing
# it explicitly here means a plain cell re-run recovers on its own.
import gc
import sys
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 either way -- '
      f'if not, the runtime was not actually restarted and still holds an earlier '
      f'crash alive; use Runtime > Restart session, not just re-running this cell)')

from omar_pfem.data.mesh_convergence_B8_prototype import solve_case, compare_to_reference

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
os.makedirs(f'{R}/b8', exist_ok=True)

# CPU-scale rows already solved and reported locally (kept in the ladder
# so the full trend, low-to-high resolution, stays visible in one plot) --
# resolutions only, re-solved here on GPU for a consistent per-row time
# basis, not reused from the earlier CPU run.
CPU_SCALE_RESOLUTIONS = [(9, 5), (13, 7), (17, 9), (21, 11), (25, 13)]
# GPU-scale ladder into the 10^5-10^6-element range the advisor asked about.
GPU_RESOLUTIONS = [(35, 18), (49, 25), (69, 35), (97, 49), (125, 63)]
RESOLUTIONS = CPU_SCALE_RESOLUTIONS + GPU_RESOLUTIONS
# nz_per_rubber/nz_per_shim scaled together with Ntheta/Nr so the mesh
# refines roughly proportionally in every direction, not just in-plane.
NZ_PARAMS = {
    (9, 5): (3, 2), (13, 7): (4, 2), (17, 9): (3, 2), (21, 11): (4, 2), (25, 13): (4, 2),
    (35, 18): (6, 3), (49, 25): (8, 4), (69, 35): (11, 5), (97, 49): (15, 7), (125, 63): (19, 9),
}
OLD_FINE_RESOLUTION = (137, 69)   # ~1,054,272 elements -- just above the 10^5-10^6 target range
OLD_FINE_NZ = (21, 10)
# NEW_FINE_RESOLUTION history, both confirmed directly on a real 80GB
# A100 (not theorized): (193,97, ~2,912,256 el) OOM'd during basic
# shape-function setup; (157,79, ~1,569,672 el) OOM'd later, during the
# first Newton iteration's stiffness assembly, needing ~6.7GB more when
# ~77.6GB was already in use -- the OLD-vs-NEW cleanup between solves
# was confirmed WORKING this time (memory returned to baseline after
# OLD), so this second failure is a genuinely different, simpler cause:
# one intermediate tensor per Newton iteration (the local element
# stiffness contribution, shape (n_elem, n_gauss=8, 24, 24) in float64)
# scales as n_elements * 3.6864e-5 GB -- ~57.9GB by itself at 1,569,672
# elements. Backing out the real numbers from that failure (total
# attempted ~84.4GB, of which ~57.9GB was this one tensor) gives ~26.5GB
# for everything else (K matrix, CG buffers, mesh tensors); targeting a
# safe ~72GB total gives a real, computed ceiling of ~1.23M elements --
# NEW_FINE_RESOLUTION below (1,254,528 el) is chosen just under that,
# not another guess.
NEW_FINE_RESOLUTION = (145, 73)   # ~1,254,528 elements -- meaningfully finer, for the reference check
NEW_FINE_NZ = (22, 11)

figs_saved = []


def save_and_show(fig, name):
    path = f'{R}/b8/{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    figs_saved.append(path)
    print('Saved figure:', path)
    plt.show()


print('\nSolving the OLD fine reference (137,69, ~1,054,272 elements)...')
ref_old = solve_case(*OLD_FINE_RESOLUTION, nz_per_rubber=OLD_FINE_NZ[0], nz_per_shim=OLD_FINE_NZ[1], device=device, verbose=True)
print(f"  OLD reference: {ref_old['n_elements']} elements, {ref_old['elapsed_s']:.2f}s, "
      f"n_region={ref_old['n_region']}, region_avg_sigma_xx={ref_old['region_avg_sigma_xx']:.4f} "
      f"(true_max={ref_old['region_true_max_sigma_xx']:.4f}, diagnostic only)")

# Confirmed directly on a real 80GB A100: GPU memory from the OLD solve
# above was NOT released before the next large solve started (the
# CUDA OOM report showed ~77GB still "allocated," not just cached, right
# before a NEW solve's own tiny first allocation) -- torch.no_grad() (in
# mesh_convergence_B8.py's own solve_case) fixes growth WITHIN one solve
# across its own load increments, but not retention ACROSS separate
# solve_case() calls. gc.collect()+empty_cache() here is the standard,
# safe fix for that (frees memory, does not change any numbers).
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

print(f'\nSolving the NEW, finer reference {NEW_FINE_RESOLUTION} (~1,254,528 elements) -- '
      'this is the check for whether the OLD reference is actually converged...')
ref_new = solve_case(*NEW_FINE_RESOLUTION, nz_per_rubber=NEW_FINE_NZ[0], nz_per_shim=NEW_FINE_NZ[1], device=device, verbose=True)
print(f"  NEW reference: {ref_new['n_elements']} elements, {ref_new['elapsed_s']:.2f}s, "
      f"n_region={ref_new['n_region']}, region_avg_sigma_xx={ref_new['region_avg_sigma_xx']:.4f} "
      f"(true_max={ref_new['region_true_max_sigma_xx']:.4f}, diagnostic only)")

print('\n' + '=' * 90)
print('OLD vs NEW reference -- the check for whether the chosen reference is '
      'actually converged (region-Cauchy field error is the PRIMARY comparison; '
      'region_avg relative change is a secondary scalar cross-check):')
d_avg = abs(ref_new['region_avg_sigma_xx'] - ref_old['region_avg_sigma_xx']) / abs(ref_old['region_avg_sigma_xx'])
print(f"  region_avg_sigma_xx: OLD={ref_old['region_avg_sigma_xx']:.4f}  "
      f"NEW={ref_new['region_avg_sigma_xx']:.4f}  relative change={d_avg*100:.3f}%")
_, cauchy_field_old_vs_new, _ = compare_to_reference(ref_old, ref_new)
print(f"  region-Cauchy FIELD error (OLD relative to NEW, PRIMARY QoI): "
      f"{cauchy_field_old_vs_new*100:.3f}%")
print(f"  (diagnostic only, NOT evidence either way) true_max_sigma_xx: "
      f"OLD={ref_old['region_true_max_sigma_xx']:.4f}  NEW={ref_new['region_true_max_sigma_xx']:.4f}")

if cauchy_field_old_vs_new < 0.10:
    print(f"\n  ==> OLD-vs-NEW region-Cauchy field error ({cauchy_field_old_vs_new*100:.3f}%) "
          f"is below 10% -- the NEW reference is reasonably converged for this "
          f"comparison; treated as the fine reference below.")
else:
    print(f"\n  ==> OLD-vs-NEW region-Cauchy field error ({cauchy_field_old_vs_new*100:.3f}%) "
          f"is still above 10% -- the reference is NOT yet demonstrated converged. "
          f"Results below against this reference should be treated as provisional, "
          f"not final, exactly as instructed.")

fig1, ax1 = plt.subplots(figsize=(6, 5))
labels = ['OLD ref\n(%s el)' % f"{ref_old['n_elements']:,}", 'NEW ref\n(%s el)' % f"{ref_new['n_elements']:,}"]
ax1.bar(labels, [ref_old['region_avg_sigma_xx'], ref_new['region_avg_sigma_xx']], color=['tab:orange', 'tab:blue'])
ax1.set_ylabel('region_avg_sigma_xx (PRIMARY scalar QoI)')
ax1.set_title(f'B8 reference-to-reference check\nregion-Cauchy field error: {cauchy_field_old_vs_new*100:.2f}%')
fig1.tight_layout()
save_and_show(fig1, 'B8_reference_check')

ref = ref_new
FINE_RESOLUTION = NEW_FINE_RESOLUTION

rows = []
for Ntheta, Nr in RESOLUTIONS:
    nzr, nzs = NZ_PARAMS[(Ntheta, Nr)]
    r = solve_case(Ntheta, Nr, nz_per_rubber=nzr, nz_per_shim=nzs, device=device)
    l2_rel, cauchy_field_rel, n_ref_region = compare_to_reference(r, ref)
    r['disp_l2_rel'] = l2_rel
    r['cauchy_field_rel'] = cauchy_field_rel
    rows.append(r)
    print(f"\n({Ntheta},{Nr})  elements={r['n_elements']:,}  time={r['elapsed_s']:.2f}s")
    print(f"  disp_L2_rel={l2_rel*100:.3f}%  "
          f"region-Cauchy FIELD error (PRIMARY)={cauchy_field_rel*100:.3f}%")
    print(f"  region(n={r['n_region']} quadrature points): "
          f"avg_sigma_xx={r['region_avg_sigma_xx']:.4f}  "
          f"(true_max={r['region_true_max_sigma_xx']:.4f}, diagnostic only, NOT used to judge convergence)")
    print(f"  equilibrium: force_rel_residual={r['force_rel_residual']:.2e}")
    gc.collect()
    torch.cuda.empty_cache()

print('\n' + '=' * 90)
print('Region-Cauchy-FIELD-error convergence across the WHOLE ladder (PRIMARY QoI):')
for r in rows:
    print(f"  n_elem={r['n_elements']:>9,}  disp_L2={r['disp_l2_rel']*100:6.2f}%  "
          f"cauchy_field={r['cauchy_field_rel']*100:6.2f}%  "
          f"true_max_sxx={r['region_true_max_sigma_xx']:>10.3f} (diagnostic)")

print('\n' + '=' * 90)
print("TARGET CHECK: does the region-Cauchy field error stay ~5-10% within the "
      "10^5-10^6-element range?")
in_target_range = [r for r in rows if 1e5 <= r['n_elements'] <= 1e6]
for r in in_target_range:
    in_band = 0.05 <= r['cauchy_field_rel'] <= 0.10
    print(f"  n_elem={r['n_elements']:>9,}  cauchy_field={r['cauchy_field_rel']*100:6.2f}%  "
          f"{'WITHIN 5-10% band' if in_band else 'OUTSIDE 5-10% band'}")
if not in_target_range:
    print("  (no tested resolution fell exactly inside 10^5-10^6 -- see the full "
          "ladder above/figure below for the surrounding trend)")

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13, 5.5))
n_elem_all = [r['n_elements'] for r in rows] + [ref_old['n_elements'], ref_new['n_elements']]
cauchy_all = [r['cauchy_field_rel'] * 100 for r in rows] + [0.0, 0.0]  # references have no self-error
ax2a.loglog([r['n_elements'] for r in rows], [r['cauchy_field_rel'] * 100 for r in rows],
            'o-', color='tab:blue', label='region-Cauchy field error (PRIMARY)')
ax2a.loglog([r['n_elements'] for r in rows], [r['disp_l2_rel'] * 100 for r in rows],
            's--', color='tab:green', label='displacement L2 error')
ax2a.axhspan(5, 10, color='gold', alpha=0.25, label='advisor target band (5-10%)')
ax2a.axvspan(1e5, 1e6, color='gray', alpha=0.12, label='advisor target range (10^5-10^6 el)')
ax2a.axvline(ref_old['n_elements'], color='tab:orange', ls=':', label='OLD reference')
ax2a.axvline(ref_new['n_elements'], color='tab:red', ls=':', label='NEW reference')
ax2a.set_xlabel('number of elements')
ax2a.set_ylabel('relative error (%)')
ax2a.set_title('B8: PRIMARY QoI convergence vs. mesh resolution')
ax2a.legend(fontsize=8)
ax2a.grid(True, which='both', alpha=0.3)

ax2b.semilogx([r['n_elements'] for r in rows], [r['region_true_max_sigma_xx'] for r in rows],
              '^-', color='tab:purple')
ax2b.axvspan(1e5, 1e6, color='gray', alpha=0.12)
ax2b.set_xlabel('number of elements')
ax2b.set_ylabel('true_max_sigma_xx (raw peak)')
ax2b.set_title('DIAGNOSTIC ONLY -- true peak stress\n(NOT used to judge convergence or difficulty)')
ax2b.grid(True, which='both', alpha=0.3)

fig2.suptitle('B8 (laminated seismic bearing) -- GPU mesh-convergence summary', fontsize=13)
fig2.tight_layout()
save_and_show(fig2, 'B8_gpu_convergence_summary')

report = {
    'resolutions': RESOLUTIONS,
    'old_fine_resolution': OLD_FINE_RESOLUTION, 'new_fine_resolution': NEW_FINE_RESOLUTION,
    'old_vs_new_region_avg_rel_change': d_avg,
    'old_vs_new_cauchy_field_rel': cauchy_field_old_vs_new,
    'rows': [{k: v for k, v in r.items() if not k.startswith('_')} for r in rows],
    'old_fine_reference': {k: v for k, v in ref_old.items() if not k.startswith('_')},
    'new_fine_reference': {k: v for k, v in ref_new.items() if not k.startswith('_')},
    'figures_saved': figs_saved,
}
out_json = f'{R}/b8/mesh_convergence_extended.json'
with open(out_json, 'w') as f:
    json.dump(report, f, indent=2, default=lambda x: x.tolist() if hasattr(x, 'tolist') else str(x))
print('\nSaved:', out_json)

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(f'{R}/b8', kind='b8_gpu_mesh_convergence',
                    args={'resolutions': RESOLUTIONS, 'old_fine_resolution': OLD_FINE_RESOLUTION,
                          'new_fine_resolution': NEW_FINE_RESOLUTION},
                    started_at=_started,
                    results={'n_rows': len(rows), 'old_vs_new_cauchy_field_rel': cauchy_field_old_vs_new},
                    outputs=[out_json] + figs_saved,
                    notes="B8 (laminated seismic bearing) GPU mesh-convergence study: real "
                          "resolution ladder into the 10^5-10^6-element range, with a "
                          "reference-to-reference check (OLD ~1.05M vs NEW ~2.9M elements) "
                          "before trusting either as a fine reference. Region-Cauchy FIELD "
                          "error is the primary local QoI throughout; true_max is diagnostic "
                          "only and was NOT used to argue this design is harder than B3.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone.')
